[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_SegmentAnything.ipynb)


# セグメンテーションデモ（Segment Anything）

カメラや写真の中から「どこに何があるか」を色分けするデモです．  
Meta の **Segment Anything Model（SAM）** を使い，画像全体を自動分割したり，クリックした場所の物体だけを切り出したりできます．

**実行環境**: Google Colab（ランタイム → GPU: **T4** 推奨）

## セルの進め方
1. **設定**（Webカメラの左右反転・モデルサイズなど）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル画像のダウンロード**
4. **Gradio の起動**

> 初回はモデル（約 375MB）のダウンロードに時間がかかります．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- カメラ映像が左右反転して見える場合は，次のセルの `MIRROR_WEBCAM` を切り替えてください（`True` = ミラー，`False` = 反転なし）．
- T4（VRAM 約 14GB）では既定の `vit_b` を推奨します．`vit_l` / `vit_h` は精度は上がりますが重いです．
- 変更後は **Gradio 起動セル**を再実行してください．


In [ ]:
# Webカメラの左右反転（ミラー表示）
# True  : 左右反転する（Gradio のデフォルトに近い自撮り表示）
# False : 左右反転しない
MIRROR_WEBCAM = True

# SAM のバックボーン（小さいほど速い・軽い）
#   vit_b : T4 向け推奨（約 375MB）
#   vit_l : より高精度・重い
#   vit_h : 最高精度・非常に重い（T4 では非推奨）
SAM_MODEL_TYPE = "vit_b"

# 自動分割の密度（大きいほど細かく・遅い）．既定 16 はデモ向け
POINTS_PER_SIDE = 16

# 推論前に長辺をこのピクセル以下へ縮小（VRAM・速度のバランス）
MAX_IMAGE_SIDE = 1024

print(f"MIRROR_WEBCAM = {MIRROR_WEBCAM}")
print(f"SAM_MODEL_TYPE = {SAM_MODEL_TYPE}")
print(f"POINTS_PER_SIDE = {POINTS_PER_SIDE}")
print(f"MAX_IMAGE_SIDE = {MAX_IMAGE_SIDE}")


## 1. ライブラリのインストール


In [ ]:
# Meta 公式 Segment Anything（Colab 標準の torch / gradio / opencv / Pillow を利用）
!pip install -q git+https://github.com/facebookresearch/segment-anything.git


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル画像をインターネットからダウンロードし，SAM のチェックポイントを取得してモデルを準備します．  
初回はモデルのダウンロードに数分かかることがあります．


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import cv2
import gradio as gr
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from segment_anything import (
    SamAutomaticMaskGenerator,
    SamPredictor,
    sam_model_registry,
)
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 定数・サンプル画像 URL・チェックポイント
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_segment_anything")
FONT_DIR = Path("fonts")
FONT_PATH = FONT_DIR / "NotoSansJP-VF.ttf"
FONT_URL = (
    "https://raw.githubusercontent.com/googlefonts/noto-cjk/main/"
    "Sans/Variable/TTF/Subset/NotoSansJP-VF.ttf"
)
CHECKPOINT_DIR = Path("models")

CHECKPOINT_URLS: dict[str, str] = {
    "vit_b": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
    "vit_l": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth",
    "vit_h": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
}

CHECKPOINT_FILENAMES: dict[str, str] = {
    "vit_b": "sam_vit_b_01ec64.pth",
    "vit_l": "sam_vit_l_0b3195.pth",
    "vit_h": "sam_vit_h_4b8939.pth",
}

# 公開画像（Wikimedia / Pexels）．人物・動物・物体など分割しやすい題材．
SAMPLE_IMAGE_SOURCES: list[tuple[str, str, str]] = [
    (
        "dog.jpg",
        "https://images.pexels.com/photos/1108099/pexels-photo-1108099.jpeg?auto=compress&cs=tinysrgb&w=800",
        "犬",
    ),
    (
        "cat.jpg",
        "https://images.pexels.com/photos/45201/kitty-cat-kitten-pet-45201.jpeg?auto=compress&cs=tinysrgb&w=800",
        "猫",
    ),
    (
        "person_asian.jpg",
        "https://images.pexels.com/photos/1239291/pexels-photo-1239291.jpeg?auto=compress&cs=tinysrgb&w=800",
        "人物（アジア系）",
    ),
    (
        "japanese_smile.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/b/b0/Smiling_Japanese_Woman.jpg",
        "人物（日本人）",
    ),
    (
        "fruits.jpg",
        "https://images.pexels.com/photos/1132047/pexels-photo-1132047.jpeg?auto=compress&cs=tinysrgb&w=800",
        "果物",
    ),
    (
        "bicycle.jpg",
        "https://images.pexels.com/photos/100582/pexels-photo-100582.jpeg?auto=compress&cs=tinysrgb&w=800",
        "自転車",
    ),
]

USER_AGENT = (
    "Mozilla/5.0 (compatible; OpenCampusDemo/1.0; "
    "+https://github.com/yryo1005/OpenCampus_Demo)"
)

# クリック点の状態（同一画像での連続クリック用）
_click_state: dict = {
    "image_id": None,
    "points": [],  # list[tuple[int, int]] 前景点
}


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（かなり時間がかかります）．")
    return "cpu"


def download_bytes(url: str, save_path: Path) -> Path:
    """URL からバイナリを取得して保存する（既存ならスキップ）．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=300) as response:
        save_path.write_bytes(response.read())
    return save_path


def download_image(url: str, save_path: Path, max_side: int = 1280) -> Path:
    """URL から画像を取得し，長辺を制限して保存する．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス
        max_side (int): 長辺の上限ピクセル（既定 1280）

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=60) as response:
        raw = response.read()
    arr = np.frombuffer(raw, dtype=np.uint8)
    bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError(f"画像のデコードに失敗しました: {url}")
    h, w = bgr.shape[:2]
    long_side = max(h, w)
    if long_side > max_side:
        scale = max_side / float(long_side)
        bgr = cv2.resize(
            bgr,
            (int(w * scale), int(h * scale)),
            interpolation=cv2.INTER_AREA,
        )
    ok, encoded = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    if not ok:
        raise RuntimeError(f"画像のエンコードに失敗しました: {save_path}")
    save_path.write_bytes(encoded.tobytes())
    return save_path


def download_font(url: str, save_path: Path) -> Path:
    """日本語表示用フォントをダウンロードする（既存ならスキップ）．

    Args:
        url (str): フォントの URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したフォントのパス
    """
    return download_bytes(url, save_path)


def prepare_sample_images(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル画像をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル画像DL", leave=False):
        path = download_image(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


def load_sam(
    model_type: str,
    device: str,
) -> tuple[torch.nn.Module, SamPredictor, SamAutomaticMaskGenerator]:
    """SAM モデル・Predictor・自動マスク生成器を構築する．

    Args:
        model_type (str): "vit_b" / "vit_l" / "vit_h"
        device (str): "cuda" または "cpu"

    Returns:
        tuple: (sam, predictor, mask_generator)
    """
    if model_type not in CHECKPOINT_URLS:
        raise ValueError(f"未知の SAM_MODEL_TYPE: {model_type}")

    ckpt_path = CHECKPOINT_DIR / CHECKPOINT_FILENAMES[model_type]
    print(f"チェックポイントを準備中: {ckpt_path.name}")
    download_bytes(CHECKPOINT_URLS[model_type], ckpt_path)
    print(f"  サイズ: {ckpt_path.stat().st_size / (1024**2):.1f} MB")

    print(f"モデルを読み込み中: {model_type} (device={device})")
    sam = sam_model_registry[model_type](checkpoint=str(ckpt_path))
    sam.to(device=device)
    sam.eval()

    predictor = SamPredictor(sam)
    mask_generator = SamAutomaticMaskGenerator(
        model=sam,
        points_per_side=int(POINTS_PER_SIDE),
        pred_iou_thresh=0.86,
        stability_score_thresh=0.92,
        crop_n_layers=0,
        min_mask_region_area=100,
    )
    return sam, predictor, mask_generator


def to_rgb_uint8(image) -> np.ndarray | None:
    """Gradio / PIL / ndarray 入力を RGB uint8 (H, W, 3) に揃える．

    Args:
        image: Gradio Image の入力（None / PIL.Image / np.ndarray）

    Returns:
        np.ndarray | None: RGB 画像．入力が無い場合は None
    """
    if image is None:
        return None
    if isinstance(image, Image.Image):
        return np.asarray(image.convert("RGB"))
    arr = np.asarray(image)
    if arr.ndim == 2:
        return cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_GRAY2RGB)
    if arr.shape[2] == 4:
        return arr[:, :, :3].astype(np.uint8)
    return arr.astype(np.uint8)


def resize_long_side(rgb: np.ndarray, max_side: int) -> np.ndarray:
    """長辺が max_side を超える場合に縮小する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        max_side (int): 長辺の上限

    Returns:
        np.ndarray: 縮小後（またはそのまま）の RGB 画像，形状 (H', W', 3)
    """
    h, w = rgb.shape[:2]
    long_side = max(h, w)
    if long_side <= max_side:
        return rgb
    scale = max_side / float(long_side)
    return cv2.resize(
        rgb,
        (int(w * scale), int(h * scale)),
        interpolation=cv2.INTER_AREA,
    )


def image_fingerprint(rgb: np.ndarray) -> tuple:
    """画像同一性の簡易指紋を返す（クリック状態のリセット判定用）．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)

    Returns:
        tuple: (高さ, 幅, 中央付近画素の合計)
    """
    h, w = rgb.shape[:2]
    cy, cx = h // 2, w // 2
    patch = rgb[max(0, cy - 2) : cy + 3, max(0, cx - 2) : cx + 3]
    return (h, w, int(patch.sum()))


def overlay_masks(
    rgb: np.ndarray,
    masks: list[np.ndarray],
    alpha: float = 0.45,
    seed: int = 42,
) -> np.ndarray:
    """複数マスクを半透明の色で重ね描画する．

    Args:
        rgb (np.ndarray): 元画像 RGB，形状 (H, W, 3)
        masks (list[np.ndarray]): 各要素が bool / uint8 のマスク，形状 (H, W)
        alpha (float): マスクの不透明度（0〜1）
        seed (int): 色の乱数シード

    Returns:
        np.ndarray: 可視化 RGB 画像，形状 (H, W, 3)
    """
    vis = rgb.astype(np.float32).copy()
    rng = np.random.default_rng(seed)
    for mask in masks:
        m = mask.astype(bool)
        if not np.any(m):
            continue
        color = rng.integers(40, 255, size=(3,), dtype=np.int32).astype(np.float32)
        vis[m] = vis[m] * (1.0 - alpha) + color * alpha
        # 輪郭を少し強調
        contours, _ = cv2.findContours(
            m.astype(np.uint8),
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE,
        )
        cv2.drawContours(vis, contours, -1, color.tolist(), 2)
    return np.clip(vis, 0, 255).astype(np.uint8)


def make_cutout(rgb: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """マスク領域だけを残し，背景を白にした切り抜き画像を返す．

    Args:
        rgb (np.ndarray): 元画像 RGB，形状 (H, W, 3)
        mask (np.ndarray): マスク，形状 (H, W)

    Returns:
        np.ndarray: 切り抜き RGB 画像，形状 (H, W, 3)
    """
    m = mask.astype(bool)
    out = np.full_like(rgb, 255)
    out[m] = rgb[m]
    return out


def draw_points_on_image(
    rgb: np.ndarray,
    points: list[tuple[int, int]],
) -> np.ndarray:
    """クリック点を緑の丸で描画する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        points (list[tuple[int, int]]): (x, y) のリスト

    Returns:
        np.ndarray: 描画後の RGB 画像，形状 (H, W, 3)
    """
    vis = rgb.copy()
    for x, y in points:
        cv2.circle(vis, (int(x), int(y)), 8, (0, 220, 80), -1)
        cv2.circle(vis, (int(x), int(y)), 10, (255, 255, 255), 2)
    return vis


def draw_status_label(rgb: np.ndarray, text: str) -> np.ndarray:
    """画像左上に日本語ステータスを描画する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        text (str): 表示文字列

    Returns:
        np.ndarray: 描画後の RGB 画像，形状 (H, W, 3)
    """
    pil = Image.fromarray(rgb)
    draw = ImageDraw.Draw(pil)
    try:
        font = ImageFont.truetype(str(FONT_PATH), 40)
    except OSError:
        font = ImageFont.load_default()
    draw.rectangle((4, 4, 4 + 26 * len(text) + 24, 64), fill=(0, 0, 0, 160))
    draw.text((12, 14), text, fill=(255, 255, 255), font=font)
    return np.asarray(pil)


def format_auto_summary(mask_dicts: list[dict]) -> str:
    """自動分割結果の説明文を作る．

    Args:
        mask_dicts (list[dict]): SamAutomaticMaskGenerator の出力

    Returns:
        str: マスク数・面積上位の説明
    """
    n = len(mask_dicts)
    if n == 0:
        return "マスクが見つかりませんでした．別の写真を試してください．"
    areas = [int(d["area"]) for d in mask_dicts]
    areas_sorted = sorted(areas, reverse=True)
    top = ", ".join(str(a) for a in areas_sorted[:5])
    lines = [
        f"検出マスク数: {n}",
        f"面積上位（ピクセル）: {top}",
        "色のついた領域が，AI が分けた「かたまり」です．",
    ]
    return "\n".join(lines)


def format_point_summary(score: float, n_points: int) -> str:
    """点指定分割の説明文を作る．

    Args:
        score (float): 予測スコア（0〜1）
        n_points (int): 使用したクリック点数

    Returns:
        str: 説明テキスト
    """
    return (
        f"クリック点数: {n_points}\n"
        f"確信度スコア: {score * 100:.1f}%\n"
        "右側は切り抜き結果です．別の場所をクリックすると点が追加されます．\n"
        "画像を変えたあと，または「クリック点をリセット」で点を消してください．"
    )


def ensure_predictor_image(rgb: np.ndarray) -> None:
    """Predictor に画像埋め込みをセットする（同一画像なら再計算しない）．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
    """
    fid = image_fingerprint(rgb)
    if getattr(predictor, "_oc_fid", None) != fid:
        predictor.set_image(rgb)
        predictor._oc_fid = fid  # type: ignore[attr-defined]


def run_automatic(rgb: np.ndarray, alpha: float) -> tuple[np.ndarray, np.ndarray, str]:
    """画像全体を自動分割する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        alpha (float): オーバーレイ不透明度

    Returns:
        tuple: (可視化画像, 最大マスクの切り抜き, 説明文)
            各画像の形状は (H, W, 3)
    """
    # 進捗表示（1 枚でもバーが出ると待ち時間が分かりやすい）
    for _ in tqdm(range(1), desc="自動分割", leave=False):
        mask_dicts = mask_generator.generate(rgb)

    mask_dicts = sorted(mask_dicts, key=lambda d: int(d["area"]), reverse=True)
    masks = [d["segmentation"] for d in mask_dicts]
    vis = overlay_masks(rgb, masks, alpha=alpha)
    vis = draw_status_label(vis, f"自動分割: {len(masks)} 個")

    if masks:
        cutout = make_cutout(rgb, masks[0])
        cutout = draw_status_label(cutout, "最大領域の切り抜き")
    else:
        cutout = rgb.copy()

    return vis, cutout, format_auto_summary(mask_dicts)


def run_point_prompt(
    rgb: np.ndarray,
    points: list[tuple[int, int]],
    alpha: float,
) -> tuple[np.ndarray, np.ndarray, str]:
    """クリック点を前景プロンプトとして物体を分割する．

    Args:
        rgb (np.ndarray): RGB 画像，形状 (H, W, 3)
        points (list[tuple[int, int]]): 前景点 (x, y)
        alpha (float): オーバーレイ不透明度

    Returns:
        tuple: (可視化画像, 切り抜き画像, 説明文)
    """
    if not points:
        blank = draw_status_label(rgb.copy(), "画像をクリックしてください")
        return blank, blank, "点指定モードです．左側の画像をクリックしてください．"

    ensure_predictor_image(rgb)
    coords = np.array(points, dtype=np.float32)
    labels = np.ones(len(points), dtype=np.int32)

    for _ in tqdm(range(1), desc="点指定分割", leave=False):
        masks, scores, _ = predictor.predict(
            point_coords=coords,
            point_labels=labels,
            multimask_output=True,
        )

    best_i = int(np.argmax(scores))
    best_mask = masks[best_i]
    best_score = float(scores[best_i])

    vis = overlay_masks(rgb, [best_mask], alpha=alpha, seed=7)
    vis = draw_points_on_image(vis, points)
    vis = draw_status_label(vis, f"点指定: スコア {best_score * 100:.0f}%")

    cutout = make_cutout(rgb, best_mask)
    cutout = draw_points_on_image(cutout, points)
    cutout = draw_status_label(cutout, "切り抜き")

    return vis, cutout, format_point_summary(best_score, len(points))


def prepare_input_image(image, mirror: bool) -> np.ndarray | None:
    """入力画像をミラー・リサイズ込みで推論用 RGB にする．

    Args:
        image: Gradio Image 入力
        mirror (bool): 左右反転するか

    Returns:
        np.ndarray | None: RGB (H, W, 3)．未入力時は None
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        return None
    if mirror:
        rgb = np.ascontiguousarray(rgb[:, ::-1, :])
    return resize_long_side(rgb, int(MAX_IMAGE_SIDE))


def map_click_to_inference(
    image,
    mirror: bool,
    click_xy: tuple[int, int],
) -> tuple[np.ndarray | None, tuple[int, int] | None]:
    """表示画像上のクリック座標を，推論用画像上の座標へ変換する．

    Gradio のクリックは左右反転・リサイズ前の表示に対応するため，
    ミラーと長辺縮小を同じ順で座標にも適用する．

    Args:
        image: Gradio Image 入力
        mirror (bool): 左右反転するか
        click_xy (tuple[int, int]): 表示上の (x, y)

    Returns:
        tuple: (推論用 RGB (H,W,3) または None, 変換後 (x, y) または None)
    """
    rgb = to_rgb_uint8(image)
    if rgb is None:
        return None, None

    x, y = int(click_xy[0]), int(click_xy[1])
    h0, w0 = rgb.shape[:2]
    x = int(np.clip(x, 0, w0 - 1))
    y = int(np.clip(y, 0, h0 - 1))

    if mirror:
        x = w0 - 1 - x
        rgb = np.ascontiguousarray(rgb[:, ::-1, :])

    rgb_small = resize_long_side(rgb, int(MAX_IMAGE_SIDE))
    h1, w1 = rgb_small.shape[:2]
    if (h0, w0) != (h1, w1):
        x = int(round(x * w1 / float(w0)))
        y = int(round(y * h1 / float(h0)))
    x = int(np.clip(x, 0, w1 - 1))
    y = int(np.clip(y, 0, h1 - 1))
    return rgb_small, (x, y)


def segment_image(
    image,
    mode: str,
    alpha: float,
    mirror: bool,
) -> tuple[np.ndarray | None, np.ndarray | None, str]:
    """Gradio 用の分割エントリ（ボタン実行）．

    Args:
        image: 入力画像
        mode (str): "自動（すべて色分け）" または "点指定（クリック）"
        alpha (float): オーバーレイ不透明度
        mirror (bool): 左右反転

    Returns:
        tuple: (可視化, 切り抜き, 説明文)
    """
    rgb = prepare_input_image(image, mirror)
    if rgb is None:
        return None, None, "画像を撮影するか，サンプル／アップロードしてください．"

    if mode.startswith("自動"):
        _click_state["image_id"] = None
        _click_state["points"] = []
        return run_automatic(rgb, float(alpha))

    # 点指定：既存の点があればそれで再推論，なければ案内
    fid = image_fingerprint(rgb)
    if _click_state["image_id"] != fid:
        _click_state["image_id"] = fid
        _click_state["points"] = []
    return run_point_prompt(rgb, list(_click_state["points"]), float(alpha))


def on_image_select(
    image,
    mode: str,
    alpha: float,
    mirror: bool,
    evt: gr.SelectData,
) -> tuple[np.ndarray | None, np.ndarray | None, str]:
    """入力画像クリック時のコールバック（点指定モード）．

    Args:
        image: 入力画像
        mode (str): モード文字列
        alpha (float): オーバーレイ不透明度
        mirror (bool): 左右反転
        evt (gr.SelectData): クリック座標（index = [x, y]）

    Returns:
        tuple: (可視化, 切り抜き, 説明文)
    """
    if not mode.startswith("点"):
        return segment_image(image, mode, alpha, mirror)

    rgb, mapped = map_click_to_inference(image, mirror, (evt.index[0], evt.index[1]))
    if rgb is None or mapped is None:
        return None, None, "画像を用意してからクリックしてください．"

    x, y = mapped
    fid = image_fingerprint(rgb)
    if _click_state["image_id"] != fid:
        _click_state["image_id"] = fid
        _click_state["points"] = []
    _click_state["points"].append((x, y))

    return run_point_prompt(rgb, list(_click_state["points"]), float(alpha))


def reset_click_points(
    image,
    mode: str,
    alpha: float,
    mirror: bool,
) -> tuple[np.ndarray | None, np.ndarray | None, str]:
    """クリック点をクリアする．

    Args:
        image: 入力画像
        mode (str): モード
        alpha (float): 不透明度
        mirror (bool): 左右反転

    Returns:
        tuple: (可視化, 切り抜き, 説明文)
    """
    _click_state["points"] = []
    rgb = prepare_input_image(image, mirror)
    if rgb is None:
        return None, None, "クリック点をリセットしました．"
    blank = draw_status_label(rgb.copy(), "クリック点をリセット")
    return blank, blank, "クリック点を消しました．点指定モードなら再度画像をクリックしてください．"


def build_demo(
    sample_items: list[tuple[str, Path]],
    mirror_webcam: bool,
) -> gr.Blocks:
    """カメラ入力・Examples・結果表示を配置した Gradio UI を構築する．

    Args:
        sample_items (list[tuple[str, Path]]): (ラベル, 画像パス)
        mirror_webcam (bool): カメラ入力を左右反転するか

    Returns:
        gr.Blocks: Gradio デモ
    """
    example_paths = [str(path) for _, path in sample_items]

    with gr.Blocks(title="セグメンテーションデモ（Segment Anything）") as demo:
        with gr.Row():
            with gr.Column(scale=1):
                image_in = gr.Image(
                    label="入力（カメラ / アップロード）",
                    type="numpy",
                    sources=["webcam", "upload"],
                    webcam_options=gr.WebcamOptions(mirror=mirror_webcam),
                )
                mode = gr.Radio(
                    choices=["自動（すべて色分け）", "点指定（クリック）"],
                    value="自動（すべて色分け）",
                    label="モード",
                )
                alpha = gr.Slider(
                    minimum=0.2,
                    maximum=0.8,
                    value=0.45,
                    step=0.05,
                    label="色の濃さ",
                )
                mirror = gr.Checkbox(
                    label="入力画像を左右反転して分割する",
                    value=False,
                    info="アップロード画像の向きが逆のときだけオンにしてください（カメラは上のミラー設定を利用）",
                )
                with gr.Row():
                    btn_run = gr.Button("セグメンテーション実行", variant="primary")
                    btn_reset = gr.Button("クリック点をリセット")

            with gr.Column(scale=1):
                image_out = gr.Image(label="分割結果", type="numpy")
                image_cut = gr.Image(label="切り抜き", type="numpy")
                text_out = gr.Textbox(label="説明", lines=5)

        gr.Examples(
            examples=example_paths,
            inputs=[image_in],
            label="サンプル画像（撮影しなくても試せます）",
        )

        inputs = [image_in, mode, alpha, mirror]
        outputs = [image_out, image_cut, text_out]

        btn_run.click(fn=segment_image, inputs=inputs, outputs=outputs)
        btn_reset.click(fn=reset_click_points, inputs=inputs, outputs=outputs)
        image_in.select(fn=on_image_select, inputs=inputs, outputs=outputs)

    return demo

# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
DEVICE = resolve_device()
download_font(FONT_URL, FONT_PATH)
sample_items = prepare_sample_images(SAMPLE_IMAGE_SOURCES, SAMPLE_DIR)
sam_model, predictor, mask_generator = load_sam(SAM_MODEL_TYPE, DEVICE)
print("準備完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

UI が起動したら，サンプル画像をクリックするか，カメラで撮影して「セグメンテーション実行」を押してください．  
**点指定**モードでは，左側の画像を直接クリックするとその場所の物体を切り出します．


In [ ]:
demo = build_demo(sample_items, mirror_webcam=MIRROR_WEBCAM)
demo.launch(share=True, debug=False)
